# SalesLuv 딜 승패 모델 3차 임계값 조정

2차에서 선택한 모델과 확률값은 그대로 두고, Train OOF 확률에서 운영 임계값을 조정합니다.
Test에는 선택된 임계값을 마지막에 적용해 분류 결과를 확인합니다.


## 1. 2차 결과 불러오기

2차 노트북이 저장한 확률값과 데이터 순서를 확인합니다.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
)

SOURCE_SHA256 = "8dee635b95bdcb00896b654efe62fc20177090081c81ef5224e8641ba31c3061"
FEATURE_NAMES = (
    "Product",
    "Seller",
    "Authority",
    "Comp_size",
    "Competitors",
    "Purch_dept",
    "Partnership",
    "Budgt_alloc",
    "Forml_tend",
    "RFI",
    "RFP",
    "Growth",
    "Posit_statm",
    "Source",
    "Client",
    "Scope",
    "Strat_deal",
    "Cross_sale",
    "Up_sale",
    "Deal_type",
    "Needs_def",
    "Att_t_client",
)
THRESHOLD_INPUT_PATH = Path("../pipeline/artifacts/deal-model-phase2-predictions.npz").resolve()

assert THRESHOLD_INPUT_PATH.exists(), "2차 노트북을 먼저 실행해야 합니다."
with np.load(THRESHOLD_INPUT_PATH, allow_pickle=False) as saved:
    schema_version = int(saved["schema_version"].item())
    source_sha256 = str(saved["source_sha256"].item())
    selected_model_name = str(saved["selected_model_name"].item())
    selected_cv_brier = float(saved["selected_cv_brier"].item())
    random_state = int(saved["random_state"].item())
    feature_names = tuple(saved["feature_names"].tolist())
    train_index = saved["train_index"].astype(np.int64)
    test_index = saved["test_index"].astype(np.int64)
    y_train = saved["y_train"].astype(np.int64)
    y_test = saved["y_test"].astype(np.int64)
    oof_probabilities = saved["oof_won_probability"].astype(float)
    test_probabilities = saved["test_won_probability"].astype(float)

assert schema_version == 1
assert source_sha256 == SOURCE_SHA256
assert feature_names == FEATURE_NAMES
assert len(y_train) == len(oof_probabilities) == len(train_index)
assert len(y_test) == len(test_probabilities) == len(test_index)
assert set(train_index).isdisjoint(set(test_index))
assert set(np.unique(y_train)).issubset({0, 1})
assert set(np.unique(y_test)).issubset({0, 1})
assert np.isfinite(oof_probabilities).all()
assert np.isfinite(test_probabilities).all()
assert ((0 <= oof_probabilities) & (oof_probabilities <= 1)).all()
assert ((0 <= test_probabilities) & (test_probabilities <= 1)).all()

print(f"선택 모델: {selected_model_name}")
print(f"CV Brier: {selected_cv_brier:.6f}")
print(f"Train OOF: {len(y_train)}건")
print(f"Test: {len(y_test)}건")

선택 모델: SoftVoting_LR_TabICL
CV Brier: 0.160846
Train OOF: 255건
Test: 110건


## 2. Train OOF 임계값 확인

임계값별 Accuracy·Precision·Recall·FP·FN을 확인합니다.
운영 임계값은 비교 결과를 바탕으로 0.50을 사용합니다.


In [2]:
threshold_values = np.arange(0.05, 1.00, 0.01)


def calculate_threshold_metrics(y_true, probabilities, threshold):
    """주어진 임계값의 분류 성능과 오분류 개수를 계산합니다."""
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions,
        labels=[0, 1],
    ).ravel()
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, predictions)),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall": float(recall_score(y_true, predictions, zero_division=0)),
        "fpr": float(fp / (fp + tn)),
        "fp": int(fp),
        "fn": int(fn),
    }


threshold_comparison = pd.DataFrame(
    [
        calculate_threshold_metrics(y_train, oof_probabilities, threshold)
        for threshold in threshold_values
    ]
)
selected_threshold = 0.50
comparison_rows = threshold_comparison[threshold_comparison["threshold"].between(0.45, 0.65)]

display(comparison_rows)
print(f"운영 임계값: {selected_threshold:.2f}")

,threshold,accuracy,precision,recall,fpr,fp,fn
40,0.45,0.768627,0.731343,0.809917,0.268657,36,23
41,0.46,0.776471,0.742424,0.809917,0.253731,34,23
42,0.47,0.772549,0.740458,0.801653,0.253731,34,24
43,0.48,0.780392,0.751938,0.801653,0.238806,32,24
44,0.49,0.772549,0.748031,0.785124,0.238806,32,26
45,0.50,0.772549,0.748031,0.785124,0.238806,32,26
46,0.51,0.772549,0.748031,0.785124,0.238806,32,26
47,0.52,0.768627,0.746032,0.776860,0.238806,32,27
48,0.53,0.768627,0.750000,0.768595,0.231343,31,28
49,0.54,0.768627,0.754098,0.760331,0.223881,30,29


운영 임계값: 0.50


## 3. Test 결과 확인

운영 임계값 0.50을 Test에 적용한 분류 결과를 확인합니다.


In [3]:
threshold_test_result = pd.DataFrame(
    [
        calculate_threshold_metrics(
            y_test,
            test_probabilities,
            selected_threshold,
        )
    ]
)

display(threshold_test_result)
selected_test_predictions = (test_probabilities >= selected_threshold).astype(int)
print("Confusion Matrix")
print(confusion_matrix(y_test, selected_test_predictions))
print("Classification Report")
print(classification_report(y_test, selected_test_predictions))

,threshold,accuracy,precision,recall,fpr,fp,fn
0,0.5,0.790909,0.745763,0.846154,0.258621,15,8


Confusion Matrix
[[43 15]
 [ 8 44]]
Classification Report
              precision    recall  f1-score   support

           0       0.84      0.74      0.79        58
           1       0.75      0.85      0.79        52

    accuracy                           0.79       110
   macro avg       0.79      0.79      0.79       110
weighted avg       0.80      0.79      0.79       110

